In [27]:
import sys
# Add the desired directory to sys.path
sys.path.append('build/')

import randSVD
from sklearn.utils.extmath import randomized_svd

seed = 7050
tol = 1e-3

## Test matrix

In [28]:
import numpy as np

class SymmetricTestMatrix:
    def __init__(self, size, eig_val_decay):
        self.rows = size
        self.cols = size
        self.eig_val_decay = eig_val_decay
        
        # Calculate rank
        self.rank = size
        
        # Generate random U matrix
        self.U = np.random.rand(size, self.rank)
        
        # Orthogonalize U using QR decomposition
        self.U, _ = np.linalg.qr(self.U)
        
        # Set the singular values based on the decay option
        if self.eig_val_decay == "fast":
            self.eig_vals = np.power(0.95, np.arange(self.rank))
        else:
            self.eig_vals = np.log(2 + np.arange(self.rank))

        self.A = (self.U @ np.diag(self.eig_vals) @ self.U.T).astype(np.float64)
        
    def matrixU(self):
        return self.U
    
    def eigenValues(self):
        return self.eig_vals
    
    def matrixA(self):
        return self.A
    

## Error metrics

In [29]:
def compute_errors(test_matrix,U,sing_vals,V):
    errors = {}

    rank = sing_vals.size

    errors["rec_error"] = np.linalg.norm(test_matrix.matrixA() - U@np.diag(sing_vals)@V.T, 'fro')
    errors["eig_val_error"] = np.linalg.norm(test_matrix.eigenValues()[:rank] - sing_vals)
    errors["eig_vect_error"] = min(max(np.linalg.norm(test_matrix.matrixU()[:,:rank]-U,axis=0)),max(np.linalg.norm(test_matrix.matrixU()[:,:rank]+U,axis=0)))
    
    return errors

## Computations

In [30]:
import pandas as pd
# Assuming randSVD and TestMatrix are already defined somewhere

test_results = pd.DataFrame(columns=['svd','size','decay','replica','rec_error', 'eig_val_error', 'eig_vect_error'])
tested_sizes = [1000]

rank = 10 #to compare with rbki
n_iter = 10
tol = 1e-10
seed = 7050

n_replicas = 10

rsi = randSVD.RSI(seed, tol)
rbki = randSVD.RBKI(seed, tol)
nys_rsi = randSVD.NysRSI(seed, tol)
nys_rbki = randSVD.NysRBKI(seed, tol)

for test_size in tested_sizes:
    print("Size: ",test_size)
    for decay in ['fast','slow']:
        print("Decay:", decay)
        test_matrix = SymmetricTestMatrix(test_size, decay)
        
        U, S, Vh = np.linalg.svd(test_matrix.matrixA(), full_matrices=False)
        optimal_error = compute_errors(test_matrix, U[:,:rank], S[:rank],Vh.T[:,:rank])

        test_results = pd.concat([test_results, pd.DataFrame([["exact", test_size,decay,i]], columns=['svd','size','decay','replica'])], ignore_index=True)
        test_results.loc[(test_results[['svd', 'size','decay','replica']] == ['exact',test_size,decay,i]).all(axis=1), 
                             ['rec_error', 'eig_val_error', 'eig_vect_error']] = list(optimal_error.values())

        for i in range(1,n_replicas+1):

            # RSI computation
            rsi.compute(test_matrix.matrixA(), rank, n_iter)
            test_results = pd.concat([test_results, pd.DataFrame([["rsi", test_size,decay,i]], columns=['svd','size','decay','replica'])], ignore_index=True)
            errors_rsi = compute_errors(test_matrix, rsi.matrixU(), rsi.singularValues(),rsi.matrixV())
            test_results.loc[(test_results[['svd', 'size','decay','replica']] == ['rsi',test_size,decay,i]).all(axis=1), 
                             ['rec_error', 'eig_val_error', 'eig_vect_error']] = list(errors_rsi.values())
            
            # RBKI computation
            rbki.compute(test_matrix.matrixA(), rank, n_iter)
            test_results = pd.concat([test_results, pd.DataFrame([["rbki", test_size,decay,i]], columns=['svd','size','decay','replica'])], ignore_index=True)
            errors_rbki = compute_errors(test_matrix, rbki.matrixU(), rbki.singularValues(),rbki.matrixV())
            test_results.loc[(test_results[['svd', 'size','decay','replica']] == ['rbki',test_size,decay,i]).all(axis=1), ['rec_error', 'eig_val_error', 'eig_vect_error']] = list(errors_rbki.values())
            
            # NysRSI computation
            nys_rsi.compute(test_matrix.matrixA(), rank, 2*n_iter+1)
            test_results = pd.concat([test_results, pd.DataFrame([["nys_rsi",test_size,decay,i]], columns=['svd','size','decay','replica'])], ignore_index=True)
            errors_nys_rsi = compute_errors(test_matrix, nys_rsi.matrixU(), nys_rsi.eigenValues(),nys_rsi.matrixU())
            test_results.loc[(test_results[['svd', 'size','decay','replica']] == ['nys_rsi',test_size,decay,i]).all(axis=1), ['rec_error', 'eig_val_error', 'eig_vect_error']] = list(errors_nys_rsi.values())

            # NysRBKI computation
            nys_rbki.compute(test_matrix.matrixA(), rank, 2*n_iter)
            test_results = pd.concat([test_results, pd.DataFrame([["nys_rbki", test_size,decay,i]], columns=['svd','size','decay','replica'])], ignore_index=True)
            errors_nys_rbki = compute_errors(test_matrix, nys_rbki.matrixU(), nys_rbki.eigenValues(),nys_rbki.matrixU())
            test_results.loc[(test_results[['svd', 'size','decay','replica']] == ['nys_rbki',test_size,decay,i]).all(axis=1), ['rec_error', 'eig_val_error', 'eig_vect_error']] = list(errors_nys_rbki.values())

            # Scikit-learn computation (randomized SVD)
            U, s, Vh = randomized_svd(test_matrix.matrixA(),
                                    n_components=rank,
                                    n_oversamples=rank,
                                    n_iter=n_iter,
                                    power_iteration_normalizer='QR',
                                    transpose=False,
                                    random_state=seed)
            test_results = pd.concat([test_results, pd.DataFrame([["scikit-learn", test_size,decay,i]], columns=['svd','size','decay','replica'])], ignore_index=True)
            errors_sklearn = compute_errors(test_matrix, U, s,Vh.T)
            test_results.loc[(test_results[['svd', 'size','decay','replica']] == ['scikit-learn',test_size,decay,i]).all(axis=1), ['rec_error', 'eig_val_error', 'eig_vect_error']] = list(errors_sklearn.values())

Size:  1000
Decay: fast
Decay: slow


In [31]:
print(test_results)

              svd  size decay replica   rec_error eig_val_error eig_vect_error
0           exact  1000  fast       2    1.917493           0.0            2.0
1             rsi  1000  fast       1    1.917493           0.0            2.0
2            rbki  1000  fast       1    1.917493           0.0            2.0
3         nys_rsi  1000  fast       1    1.917493           0.0            2.0
4        nys_rbki  1000  fast       1    1.917493           0.0            2.0
..            ...   ...   ...     ...         ...           ...            ...
97            rsi  1000  slow      10  188.440133     16.037384       1.414214
98           rbki  1000  slow      10  188.409274     16.303951       1.414478
99        nys_rsi  1000  slow      10  188.440568     16.025758       1.414214
100      nys_rbki  1000  slow      10  188.402781     16.358474       1.414214
101  scikit-learn  1000  slow      10  188.439577     16.042829       1.414214

[102 rows x 7 columns]


In [32]:
test_results.to_csv('results/randEVD_comparison.csv', index=False)